# Translate PCP Traces to E-PCP Vectors

In [2]:
import os
import sys
from pathlib import Path
import numpy as np
from typing import List, Tuple, Dict
import json
from tqdm import tqdm
from gensim.models import Word2Vec

# 1. Get the directory where this notebook is located (.../DL_trojan_detection/notebooks)
notebook_dir = Path.cwd()

# 2. Get the project root by going up one level (.../DL_trojan_detection)
project_root = notebook_dir.parent

# 3. Add the project root to sys.path so python can find the 'src' folder
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Check if it worked (Optional print)
print(f"Project root added to path: {project_root}")

# Now your imports will work
from src.parser.verilog_parser import VerilogParser
from src.parser.label_parser import LabelParser
from src.graph.graph_builder import GraphBuilder
from src.extraction.block_tree_extractor import BlockTreeExtractor
from src.traces.pcp_trace_generator import PCPTraceGenerator

Project root added to path: e:\University of Kerman\Term 9\VHDL\Paper Implementation\DL_trojan_detection


In [3]:
class EPCPTranslator:
    """Translate PCP traces to E-PCP vectors with labels"""
    
    def __init__(
        self,
        word2vec_model_path: str,
        dataset_root: str,
        logic_level: int = 4,
        max_trace_length: int = 9  # 2*logic_level + 1
    ):
        """
        Initialize translator
        
        Args:
            word2vec_model_path: Path to trained Word2Vec model
            dataset_root: Path to Datasets folder
            logic_level: BFS depth (should match Word2Vec training)
            max_trace_length: Maximum trace length for padding
        """
        self.dataset_root = Path(dataset_root)
        self.logic_level = logic_level
        self.max_trace_length = max_trace_length
        
        # Load Word2Vec model
        print(f"Loading Word2Vec model...")
        self.word2vec = Word2Vec.load(word2vec_model_path)
        self.vector_size = self.word2vec.wv.vector_size
        print(f"   Vocabulary: {len(self.word2vec.wv):,} words")
        print(f"   Vector size: {self.vector_size}\n")
        
        # Parsers
        self.verilog_parser = VerilogParser()
        self.label_parser = LabelParser()
        
        # Statistics
        self.stats = {
            'total_files': 0,
            'total_traces': 0,
            'ht_traces': 0,
            'normal_traces': 0,
            'oov_words': 0,  # Out-of-vocabulary
            'files_processed': []
        }
    
    def pcp_to_vector(self, pcp_word: str) -> np.ndarray:
        """
        Convert PCP word to vector
        
        Args:
            pcp_word: PCP word string
            
        Returns:
            100-dim vector (or zeros if OOV)
        """
        if pcp_word in self.word2vec.wv:
            return self.word2vec.wv[pcp_word]
        else:
            # Out-of-vocabulary: return zero vector
            self.stats['oov_words'] += 1
            return np.zeros(self.vector_size, dtype=np.float32)
    
    def trace_to_epcp(self, pcp_trace: List[str]) -> np.ndarray:
        """
        Convert PCP trace to E-PCP trace matrix
        
        Args:
            pcp_trace: List of PCP words
            
        Returns:
            Matrix of shape (max_trace_length, vector_size)
            Shorter traces are zero-padded
        """
        # Initialize with zeros (padding)
        epcp_matrix = np.zeros(
            (self.max_trace_length, self.vector_size),
            dtype=np.float32
        )
        
        # Fill in vectors for each PCP word
        for i, pcp_word in enumerate(pcp_trace[:self.max_trace_length]):
            epcp_matrix[i] = self.pcp_to_vector(pcp_word)
        
        return epcp_matrix
    
    def extract_labeled_traces(
        self,
        netlist_path: Path,
        label_path: Path
    ) -> Tuple[List[np.ndarray], List[int]]:
        """
        Extract E-PCP traces with labels from a single .v file
        
        Args:
            netlist_path: Path to .v file
            label_path: Path to log.txt
            
        Returns:
            (e_pcp_traces, labels) where labels are 0=Normal, 1=HT
        """
        try:
            # Parse netlist
            netlist_data = self.verilog_parser.parse(str(netlist_path))
            
            # Parse labels
            label_data = self.label_parser.parse(str(label_path))
            trojan_components = set(label_data['trojan_components'])
            
            # Build graph
            builder = GraphBuilder()
            graph = builder.build(netlist_data)
            
            # Extract blocks
            extractor = BlockTreeExtractor(graph, logic_level=self.logic_level)
            blocks = extractor.extract_all_blocks(verbose=False)
            
            # Generate PCP traces
            generator = PCPTraceGenerator(graph)
            generator.generate_all_traces(blocks, verbose=False)
            
            # Translate to E-PCP with labels
            e_pcp_traces = []
            labels = []
            
            for trace in generator.all_traces:
                # Get PCP words
                pcp_words = trace.to_string().split()
                
                # Translate to E-PCP
                epcp_matrix = self.trace_to_epcp(pcp_words)
                e_pcp_traces.append(epcp_matrix)
                
                # Label: 1 if center component is Trojan, 0 otherwise
                center_component = trace.center_cell  # Assuming this exists
                is_trojan = 1 if center_component in trojan_components else 0
                labels.append(is_trojan)
            
            # Update statistics
            self.stats['total_traces'] += len(labels)
            self.stats['ht_traces'] += sum(labels)
            self.stats['normal_traces'] += len(labels) - sum(labels)
            
            return e_pcp_traces, labels
            
        except Exception as e:
            print(f" Error: {e}")
            return [], []
    
    def find_verilog_files(
        self,
        datasets: List[str],
        benchmarks: List[str],
        t_id_ranges: Dict[str, List[Tuple[int, int]]]
    ) -> List[Tuple[Path, Path]]:
        """Find .v files matching criteria (same as training)"""
        verilog_files = []
        
        for dataset_name in datasets:
            dataset_path = self.dataset_root / dataset_name / dataset_name
            
            if not dataset_path.exists():
                continue
            
            ranges = t_id_ranges.get(dataset_name, [])
            
            for benchmark in benchmarks:
                for start_id, end_id in ranges:
                    for t_id in range(start_id, end_id + 1):
                        folder_name = f"{benchmark}_T{t_id:03d}"
                        folder_path = dataset_path / folder_name
                        
                        if folder_path.exists():
                            verilog_file = folder_path / f"{folder_name}.v"
                            label_file = folder_path / "log.txt"
                            
                            if verilog_file.exists() and label_file.exists():
                                verilog_files.append((verilog_file, label_file))
        
        return sorted(verilog_files, key=lambda x: x[0])
    
    def create_dataset(
        self,
        verilog_files: List[Tuple[Path, Path]],
        output_path: str
    ):
        """
        Create labeled E-PCP dataset
        
        Args:
            verilog_files: List of (netlist_path, label_path) tuples
            output_path: Output .npz file path
        """
        print("\n" + "="*80)
        print("CREATING E-PCP DATASET")
        print("="*80)
        print(f"Files to process: {len(verilog_files)}")
        print(f"Output: {output_path}\n")
        
        all_traces = []
        all_labels = []
        
        for i, (netlist_path, label_path) in enumerate(tqdm(verilog_files, desc="Processing files")):
            
            traces, labels = self.extract_labeled_traces(netlist_path, label_path)
            
            if traces:
                all_traces.extend(traces)
                all_labels.extend(labels)
                self.stats['files_processed'].append(str(netlist_path))
        
        self.stats['total_files'] = len(self.stats['files_processed'])
        
        # Convert to numpy arrays
        X = np.array(all_traces, dtype=np.float32)  # Shape: (N, max_len, 100)
        y = np.array(all_labels, dtype=np.int32)    # Shape: (N,)
        
        # Save dataset
        print(f"\n Saving dataset...")
        np.savez_compressed(
            output_path,
            X=X,
            y=y,
            stats=self.stats,
            vector_size=self.vector_size,
            max_trace_length=self.max_trace_length
        )
        
        # Save metadata
        metadata_path = output_path.replace('.npz', '_metadata.json')
        with open(metadata_path, 'w') as f:
            json.dump({
                'dataset_shape': {
                    'X': list(X.shape),
                    'y': list(y.shape)
                },
                'statistics': self.stats,
                'class_distribution': {
                    'HT': int(np.sum(y)),
                    'Normal': int(len(y) - np.sum(y)),
                    'HT_percentage': float(np.mean(y) * 100)
                }
            }, f, indent=2)
        
        self._print_summary(X, y, output_path)
    
    def _print_summary(self, X: np.ndarray, y: np.ndarray, output_path: str):
        """Print dataset creation summary"""
        print("\n" + "="*80)
        print("✅ DATASET CREATED!")
        print("="*80)
        print(f"\n Dataset Statistics:")
        print(f"  Total files: {self.stats['total_files']}")
        print(f"  Total traces: {self.stats['total_traces']:,}")
        print(f"  HT traces: {self.stats['ht_traces']:,} ({self.stats['ht_traces']/self.stats['total_traces']*100:.2f}%)")
        print(f"  Normal traces: {self.stats['normal_traces']:,} ({self.stats['normal_traces']/self.stats['total_traces']*100:.2f}%)")
        print(f"  OOV words: {self.stats['oov_words']:,}")
        
        print(f"\n Array Shapes:")
        print(f"  X: {X.shape} (num_traces, max_len, vector_size)")
        print(f"  y: {y.shape} (num_traces,)")
        
        print(f"\nSaved:")
        print(f"  Dataset: {output_path}")
        print(f"  Metadata: {output_path.replace('.npz', '_metadata.json')}")
        print("="*80 + "\n")

In [5]:
"""Create E-PCP dataset from trained Word2Vec"""

# ========================================================================
# CONFIGURATION
# ========================================================================

# Paths
WORD2VEC_MODEL = project_root / "notebooks" / "test_models" / "word2vec_final.model"
WORD2VEC_MODEL = str(WORD2VEC_MODEL)

DATASET_ROOT = r"E:\University of Kerman\Term 9\VHDL\Datasets"
OUTPUT_FILE = "e_pcp_dataset_T100.npz"

# Parameters (should match training)
LOGIC_LEVEL = 4
MAX_TRACE_LENGTH = 9  # 2*4 + 1

# Dataset selection (use SAME files as Word2Vec training)
DATASETS = ['TRIT-TC']
# BENCHMARKS = ['c2670', 'c3540', 'c5315']
BENCHMARKS = ['c2670']
T_ID_RANGES = {
    'TRIT-TC': [(0, 99)] # 'TRIT-TC': [(0, 99)]
}

# ========================================================================
# CREATE DATASET
# ========================================================================

# Check model exists
if not os.path.exists(WORD2VEC_MODEL):
    raise FileNotFoundError("Word2Vec model file is missing")

# Create translator
translator = EPCPTranslator(
    word2vec_model_path=WORD2VEC_MODEL,
    dataset_root=DATASET_ROOT,
    logic_level=LOGIC_LEVEL,
    max_trace_length=MAX_TRACE_LENGTH
)

# Find files
print("Finding Verilog files...")
verilog_files = translator.find_verilog_files(
    datasets=DATASETS,
    benchmarks=BENCHMARKS,
    t_id_ranges=T_ID_RANGES
)

if not verilog_files:
    print("No files found!")
    raise("File not found!")

print(f"   Found {len(verilog_files)} files\n")

# Create dataset
translator.create_dataset(verilog_files, OUTPUT_FILE)

print("Ready for LSTM training!\n")

Loading Word2Vec model...
   Vocabulary: 383 words
   Vector size: 100

Finding Verilog files...
   Found 100 files


CREATING E-PCP DATASET
Files to process: 100
Output: e_pcp_dataset_T100.npz



Processing files:   0%|          | 0/100 [00:00<?, ?it/s]

Parsing netlist: E:\University of Kerman\Term 9\VHDL\Datasets\TRIT-TC\TRIT-TC\c2670_T000\c2670_T000.v
Module name: c2670
Found 373 ports
Found 641 wire declarations
Parsed 691 components, 6 cell types

Building pin-level graph for module 'c2670'...
  Created 373 port vertices
  Created 691 cell vertices
  Created 1956 pin vertices
  Created 3912 cell-pin edges
  Created 1315 pin-pin edges (through nets)
✓ Graph built: 3020 vertices, 5227 edges


Processing files:   1%|          | 1/100 [00:00<01:11,  1.38it/s]

Parsing netlist: E:\University of Kerman\Term 9\VHDL\Datasets\TRIT-TC\TRIT-TC\c2670_T001\c2670_T001.v
Module name: c2670
Found 746 ports
Found 648 wire declarations
Parsed 1384 components, 8 cell types

Building pin-level graph for module 'c2670'...
  Created 746 port vertices
  Created 1384 cell vertices
  Created 3923 pin vertices
  Created 3974 cell-pin edges
  Created 1341 pin-pin edges (through nets)
✓ Graph built: 3058 vertices, 5315 edges


...

Parsing netlist: E:\University of Kerman\Term 9\VHDL\Datasets\TRIT-TC\TRIT-TC\c2670_T098\c2670_T098.v
Module name: c2670
Found 36927 ports
Found 1354 wire declarations
Parsed 68632 components, 17 cell types

Building pin-level graph for module 'c2670'...
  Created 36927 port vertices
  Created 68632 cell vertices
  Created 194298 pin vertices
  Created 9140 cell-pin edges
  Created 3454 pin-pin edges (through nets)
✓ Graph built: 6347 vertices, 12594 edges


Processing files:  99%|█████████▉| 99/100 [17:30<00:28, 28.46s/it]

Parsing netlist: E:\University of Kerman\Term 9\VHDL\Datasets\TRIT-TC\TRIT-TC\c2670_T099\c2670_T099.v
Module name: c2670
Found 37300 ports
Found 1359 wire declarations
Parsed 69323 components, 17 cell types

Building pin-level graph for module 'c2670'...
  Created 37300 port vertices
  Created 69323 cell vertices
  Created 196254 pin vertices
  Created 9180 cell-pin edges
  Created 3470 pin-pin edges (through nets)
✓ Graph built: 6372 vertices, 12650 edges


Processing files: 100%|██████████| 100/100 [18:02<00:00, 10.82s/it]



 Saving dataset...

✅ DATASET CREATED!

 Dataset Statistics:
  Total files: 100
  Total traces: 2,712,148
  HT traces: 16,934 (0.62%)
  Normal traces: 2,695,214 (99.38%)
  OOV words: 77,501

 Array Shapes:
  X: (2712148, 9, 100) (num_traces, max_len, vector_size)
  y: (2712148,) (num_traces,)

Saved:
  Dataset: e_pcp_dataset_T100.npz
  Metadata: e_pcp_dataset_T100_metadata.json

Ready for LSTM training!

